# 02 数据清洗与存储优化

本部分使用 DuckDB 对 2019 年 10 月和 11 月的电商用户行为数据进行清洗，并将原始 CSV 转换为 Parquet 格式，以降低后续查询和分析的数据读取开销。

In [2]:
import duckdb

query = """
SELECT *
FROM read_csv_auto('../data/raw/2019-Nov.csv')
LIMIT 5
"""

nov_sample = duckdb.sql(query).df()
nov_sample

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [3]:
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(event_time) AS start_time,
    MAX(event_time) AS end_time
FROM read_csv_auto('../data/raw/2019-Nov.csv')
"""

nov_info = duckdb.sql(query).df()
nov_info

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_time,end_time
0,67501979,2019-11-01,2019-11-30 23:59:59


In [4]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (event_time, event_type, product_id, user_id, user_session)) AS unique_rows
FROM read_csv_auto([
    '../data/raw/2019-Oct.csv',
    '../data/raw/2019-Nov.csv'
])
"""

duplicate_check = duckdb.sql(query).df()
duplicate_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,109950743,109819993


检查真正的完全重复记录

In [7]:
query = """
SELECT COUNT(*) AS unique_rows
FROM (
    SELECT DISTINCT *
    FROM read_csv_auto([
        '../data/raw/2019-Oct.csv',
        '../data/raw/2019-Nov.csv'
    ])
)
"""

full_unique = duckdb.sql(query).df()
full_unique

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,unique_rows
0,109820004


In [8]:
query = """
COPY (
    SELECT DISTINCT *
    FROM read_csv_auto([
        '../data/raw/2019-Oct.csv',
        '../data/raw/2019-Nov.csv'
    ])
    WHERE user_session IS NOT NULL
)
TO '../data/processed/ecommerce_clean.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

duckdb.sql(query)

print("清洗完成，Parquet 文件已保存。")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

清洗完成，Parquet 文件已保存。


In [9]:
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(event_time) AS start_time,
    MAX(event_time) AS end_time
FROM '../data/processed/ecommerce_clean.parquet'
"""

clean_info = duckdb.sql(query).df()
clean_info

,total_rows,start_time,end_time
0,109819992,2019-10-01,2019-11-30 23:59:59


## 数据清洗小结

合并 2019 年 10 月和 11 月电商行为数据后，原始数据共约 1.10 亿条记录。使用 DuckDB 完成重复数据处理和缺失 session 记录过滤，并将清洗结果保存为 Parquet 格式。

最终得到 109,819,992 条有效行为记录，数据时间范围为 2019-10-01 至 2019-11-30。Parquet 文件大小约为 3.67 GB，相比原始 CSV 显著降低了存储空间，也便于后续 SQL 查询和业务分析。